In [371]:
import pandas as pd

# Read the Excel file into a DataFrame
df = pd.read_excel("salesforce_report.xlsx")

In [372]:
# Display the first 5 rows
print(df.head(5))

            Start Date    End Date Campaign Record Type Campaign categories  \
0  2024-06-08 00:00:00  21/08/2024                Event   Tech; Fundraising   
1  2024-03-09 00:00:00  25/09/2024                Event                Tech   
2  2024-06-08 00:00:00  21/08/2024                Event   Tech; Fundraising   
3                  NaN  27/02/2024                Event                Tech   
4                  NaN  19/03/2024                Event                Tech   

                                       Campaign Name                Full Name  \
0  [Propel event] Aplica a grants con confianza (...  Valentina Medrano Coley   
1  [Workshop] Fortalece tu historia de impacto (2...  Valentina Medrano Coley   
2  [Propel event] Aplica a grants con confianza (...           Milagros Luque   
3  [Workshop] Eleva tu fundraising con ChatGPT I ...           Milagros Luque   
4       [Workshop] Visibiliza tu causa con IA (2024)           Milagros Luque   

  Primary Affiliation: Account Name   

In [373]:
# Display DataFrame info
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2002 entries, 0 to 2001
Data columns (total 19 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   Start Date                         1741 non-null   object 
 1   End Date                           1975 non-null   object 
 2   Campaign Record Type               2002 non-null   object 
 3   Campaign categories                1631 non-null   object 
 4   Campaign Name                      2002 non-null   object 
 5   Full Name                          2002 non-null   object 
 6   Primary Affiliation: Account Name  1753 non-null   object 
 7   Email                              2002 non-null   object 
 8   Billing Country                    300 non-null    object 
 9   Country Presence                   60 non-null     object 
 10  Country                            18 non-null     object 
 11  Country.1                          182 non-null    objec

In [374]:
# Value's percentage per column with missing data
missing_pct = df.isna().mean().sort_values(ascending=False) * 100
print(missing_pct)

#NAME?                               100.000000
Country                               99.100899
Country Presence                      97.002997
Social Cause                          96.353646
Country.1                             90.909091
Billing Country                       85.014985
Origin                                80.569431
Campaign categories                   18.531469
Start Date                            13.036963
Primary Affiliation: Account Name     12.437562
End Date                               1.348651
Campaign Record Type                   0.000000
Email                                  0.000000
Campaign Name                          0.000000
Full Name                              0.000000
Attended                               0.000000
Registered                             0.000000
Recibe newsletter                      0.000000
Campaign Subtype                       0.000000
dtype: float64


In [375]:
# Define threshold
threshold = 0.8

# Calculate missing value ratios
missing_ratio = df.isna().mean()

# Identify columns to drop and to keep
dropped_cols = missing_ratio[missing_ratio >= threshold].index.tolist()
kept_cols = missing_ratio[missing_ratio < threshold].index.tolist()

print("Columns dropped (missing > 80%):")
for col in dropped_cols:
    print(f" - {col} ({missing_ratio[col]*100:.2f}% missing)")

print("\nColumns kept:")
for col in kept_cols:
    print(f" - {col} ({missing_ratio[col]*100:.2f}% missing)")

# Drop the columns
df = df.loc[:, df.isna().mean() < threshold]

Columns dropped (missing > 80%):
 - Billing Country (85.01% missing)
 - Country Presence (97.00% missing)
 - Country (99.10% missing)
 - Country.1 (90.91% missing)
 - Origin (80.57% missing)
 - Social Cause (96.35% missing)
 - #NAME? (100.00% missing)

Columns kept:
 - Start Date (13.04% missing)
 - End Date (1.35% missing)
 - Campaign Record Type (0.00% missing)
 - Campaign categories (18.53% missing)
 - Campaign Name (0.00% missing)
 - Full Name (0.00% missing)
 - Primary Affiliation: Account Name (12.44% missing)
 - Email (0.00% missing)
 - Registered (0.00% missing)
 - Attended (0.00% missing)
 - Recibe newsletter (0.00% missing)
 - Campaign Subtype (0.00% missing)


In [376]:
# Checking missing data percentages again
missing_pct = df.isna().mean().sort_values(ascending=False) * 100
print(missing_pct)

Campaign categories                  18.531469
Start Date                           13.036963
Primary Affiliation: Account Name    12.437562
End Date                              1.348651
Campaign Record Type                  0.000000
Campaign Name                         0.000000
Full Name                             0.000000
Email                                 0.000000
Registered                            0.000000
Attended                              0.000000
Recibe newsletter                     0.000000
Campaign Subtype                      0.000000
dtype: float64


In [377]:
# Visualizing there are values that are not properly tagged in "Primary Affiliation: Account Name" column
col = "Primary Affiliation: Account Name"

# Define the replacements
dictionary = {
    "-": "Unknown",
    "ninguno": "None",
    "ninguna": "None"
}

# Count how many rows match any of the keys before replacement
mask = df[col].isin(dictionary.keys())
rows_before = mask.sum()

# Apply the replacements
df[col] = df[col].replace(dictionary)

print(f"Rows affected by replacement: {rows_before}")

Rows affected by replacement: 2


In [378]:
# We apply title case to the "Full Name" column, skipping the header row
df.loc[1:, "Full Name"] = df.loc[1:, "Full Name"].astype(str).str.title()

In [379]:
# Convert date columns to datetime format
date_cols = ["Start Date", "End Date"]

#  Function to parse and format dates
def parse_and_format_date(series):
   
    # Try converting to datetime with dayfirst=True
    parsed = pd.to_datetime(series, dayfirst=True, errors="coerce")

    # Convert valid dates to text in dd/mm/yyyy format
    formatted = parsed.dt.strftime("%d/%m/%Y")

    # Replace NaN with empty string
    formatted = formatted.fillna("")

    return formatted


# Apply to each column
for col in date_cols:
    if col in df.columns:
        df[col] = parse_and_format_date(df[col])
    else:
        print(f"Column not found: {col}")

In [380]:
# Columns to analyze
columns = ["Campaign Subtype", "Campaign Record Type", "Campaign categories"]

# List the value counts for each specified column
for col in columns:
    print(f"\n--- {col} ---")
    print(df[col].value_counts(dropna=False))


--- Campaign Subtype ---
Campaign Subtype
Workshop        1322
Propel event     680
Name: count, dtype: int64

--- Campaign Record Type ---
Campaign Record Type
Event    2002
Name: count, dtype: int64

--- Campaign categories ---
Campaign categories
Tech                 1170
Tech; Fundraising     461
NaN                   371
Name: count, dtype: int64


In [381]:
# Optional:
# This code blocks fills missing values with 'Unknown' for all columns facilitating readability.
def fill_missing_with_unknown(df):
    
    # Replace empty strings with NaN
    df = df.replace(r'^\s*$', pd.NA, regex=True)

    # Fill all missing or null values with 'Unknown'
    df_filled = df.fillna("Unknown")
    
    return df_filled

""""
In this case, we use the function to fill missing values for readibility purposes in the final dataset.
But if you prefer to keep missing values as NaN (usually better for analysis data), 
you can kindly skip this step.
"""
# Executing the function
df =fill_missing_with_unknown(df)


In [382]:

""""
Optional:
As well, you can convert boolean columns from 1/0 to "True"/"False" facilitating readability 
with this code block.
"""
""""

boolean_cols = ["Registered", "Attended", "Recibe newsletter"]
for col in boolean_cols:
    df[col] = df[col].map({1: "True", 0: "False"})
    
"""

'"\n\nboolean_cols = ["Registered", "Attended", "Recibe newsletter"]\nfor col in boolean_cols:\n    df[col] = df[col].map({1: "True", 0: "False"})\n\n'

In [383]:
# Final cleaned DataFrame
#cleaned_df =df.to_excel("cleaned_salesforce_report.xlsx", index=False)